In [ ]:
!pip install -U transformers==4.57.1 peft==0.20.0 datasets==4.3.0 accelerate==1.11.0 bitsandbytes==0.48.2 trl==0.23.0

In [ ]:
import torch
import importlib.metadata as md

print("Torch:", torch.__version__)
print("TorchVision:", md.version("torchvision"))
print("Transformers:", md.version("transformers"))
print("PEFT:", md.version("peft"))
print("Accelerate:", md.version("accelerate"))
print("TRL:", md.version("trl"))

In [ ]:
import sys
import torch
import transformers
import peft
import datasets
import accelerate
import bitsandbytes
import trl

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

In [ ]:
from datasets import load_dataset

In [ ]:
dataset=load_dataset(
    "json",
    data_files="/kaggle/input/datasets/shouldichangemyname/dcot-dataset/dcot_trainable.json"
)
print(dataset)
print(dataset["train"][0])

In [ ]:
print(dataset["train"].shape)

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM

In [ ]:
model_name='microsoft/phi-2'

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(model_name)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

In [ ]:
print("Tokenizer loaded")
print("Vocab size",tokenizer.vocab_size)
print("EOS Token",tokenizer.eos_token)

In [ ]:
import torch
from transformers import BitsAndBytesConfig

In [ ]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
print("phi2 model loaded successfully")
print("Model device", base_model.device)

In [ ]:
from peft import LoraConfig, get_peft_model,prepare_model_for_kbit_training

base_model=prepare_model_for_kbit_training(base_model)

lora_config=LoraConfig(
    r=64,
    lora_alpha=16,
    bias="none",
    lora_dropout=0.1,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "dense",
        "fc1",
        "fc2"
    ]
)

peft_model=get_peft_model(base_model,lora_config)
peft_model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args=TrainingArguments(
    output_dir='/kaggle/working/models/phi2-dcot',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    weight_decay=0.001,
    optim="paged_adamw_32bit",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    fp16=False,
    bf16=False,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    report_to="none",
)

print("Training config created")

In [ ]:
from trl import SFTTrainer

In [ ]:
trainer=SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

In [ ]:
print("SFT trainer created")

In [ ]:
trainer.train()

In [ ]:
print("Done training")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/phi2-dcot-final",
    "zip",
    root_dir="/kaggle/working/models/phi2-dcot",
    base_dir="checkpoint-31143"
)

print("Created: /kaggle/working/phi2-dcot-final.zip")